In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder\
    .appName('SparkSQL Example')\
    .getOrCreate()

In [2]:
listings = spark.read.csv(
    "data/listings.csv.gz",
    header=True,
    inferSchema=True,
    sep=",",
    quote='"',
    escape='"',
    multiLine=True,
    mode="PERMISSIVE"
)

In [3]:
reviews = spark.read.csv(
    "data/reviews.csv.gz",
    header=True,
    inferSchema=True,
    sep=",",
    quote='"',
    escape='"',
    multiLine=True,
    mode="PERMISSIVE"
)

In [4]:
listing_reviews = listings.join(
    reviews,
    listings.id == reviews.listing_id,
    how='inner'
)

In [7]:
reviews_per_listing = listing_reviews\
    .groupBy(listings.id, listings.name)\
    .agg(F.count(reviews.id).alias('num_reviews'))\
    .orderBy('num_reviews', ascending=False)\
    .show(truncate=False)

+--------+--------------------------------------------------+-----------+
|id      |name                                              |num_reviews|
+--------+--------------------------------------------------+-----------+
|47408549|Double Room+ Ensuite                              |1902       |
|43120947|Private double room with en suite facilities      |1647       |
|19670926|Locke Studio Apartment at Leman Locke             |1443       |
|2126708 |London's best transport hub 5 mins walk! Safe too!|1142       |
|46233904|Superior Studio, avg size 23.5 msq                |1002       |
|2659707 |Large Room + Private Bathroom, E3.                |998        |
|27833488|S - Heathrow Airport Terminal 2 3 4 5 Hatton Cross|951        |
|4748665 |Single bedroom near London Stratford              |933        |
|42081759|Micro Studio at Locke at Broken Wharf             |914        |
|5266466 |Large London Room, Ensuite Bathroom,TV & Breakfast|909        |
|2025844 |Family Friendly Central Lond

In [8]:
listings.createOrReplaceTempView("listings")
reviews.createOrReplaceTempView("reviews")

In [11]:
query = """
SELECT l.id,
    l.name,
    count(r.id) as num_reviews
FROM listings AS l
INNER JOIN reviews as r
    ON l.id == r.listing_id
GROUP BY 1, 2
ORDER BY num_reviews DESC"""

reviews_per_listing = spark.sql(query)
reviews_per_listing.show(truncate=False)

+--------+--------------------------------------------------+-----------+
|id      |name                                              |num_reviews|
+--------+--------------------------------------------------+-----------+
|47408549|Double Room+ Ensuite                              |1902       |
|43120947|Private double room with en suite facilities      |1647       |
|19670926|Locke Studio Apartment at Leman Locke             |1443       |
|2126708 |London's best transport hub 5 mins walk! Safe too!|1142       |
|46233904|Superior Studio, avg size 23.5 msq                |1002       |
|2659707 |Large Room + Private Bathroom, E3.                |998        |
|27833488|S - Heathrow Airport Terminal 2 3 4 5 Hatton Cross|951        |
|4748665 |Single bedroom near London Stratford              |933        |
|42081759|Micro Studio at Locke at Broken Wharf             |914        |
|5266466 |Large London Room, Ensuite Bathroom,TV & Breakfast|909        |
|2025844 |Family Friendly Central Lond